# Use Case 7: Multi-Tenant Namespace Isolation

**The Concept:** 
In production AI systems, multiple tenants (users, departments, organizations) must have their memory and data strictly isolated. A healthcare AI must never leak Patient A's records into Patient B's context.

**The Architecture:** 
SochDB provides a dedicated **Namespace Isolation** layer (`sochdb.memory.isolation`). Each tenant gets a `ScopedNamespace` that enforces strict boundaries. Cross-namespace data sharing is only possible via explicit **Grants** with expiry and audit trails.

---

### Step 0: Install Packages & Setup

In [1]:
!pip install sochdb openai python-dotenv

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

CHAT_MODEL = "gemini-3-flash-preview"

You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database & Namespace Manager
The `NamespaceManager` is the central authority for creating, listing, and managing isolated tenant namespaces.

In [2]:
from sochdb import Database
from sochdb.memory.isolation import NamespaceManager, NamespacePolicy

db = Database.open("./multitenant_demo_db")

# Create a namespace manager with EXPLICIT policy (allows grants between namespaces)
# STRICT policy blocks all cross-namespace access; EXPLICIT allows it via grants
manager = NamespaceManager.from_database(db, policy=NamespacePolicy.EXPLICIT)
print("Namespace Manager initialized with EXPLICIT isolation policy.")

Namespace Manager initialized with EXPLICIT isolation policy.


### Step 2: Create Isolated Tenant Namespaces
Each tenant (e.g., department or user) gets their own isolated namespace.

In [3]:
# Create isolated namespaces for different departments
hr_ns = manager.create("dept.hr", metadata={"department": "Human Resources", "admin": "alice"})
eng_ns = manager.create("dept.engineering", metadata={"department": "Engineering", "admin": "bob"})
finance_ns = manager.create("dept.finance", metadata={"department": "Finance", "admin": "carol"})

print(f"Created namespace: {hr_ns}")
print(f"Created namespace: {eng_ns}")
print(f"Created namespace: {finance_ns}")

# List all namespaces
all_ns = manager.list()
print(f"\nAll namespaces: {[str(ns) for ns in all_ns]}")

Created namespace: dept.hr
Created namespace: dept.engineering
Created namespace: dept.finance

All namespaces: ['dept.engineering', 'dept.finance', 'dept.hr']


### Step 3: Scoped Operations — Store Data Per Tenant
Each department stores facts in its own isolated namespace using the database's built-in namespace system. Data never leaks across scopes.

In [4]:
# Use SochDB's built-in namespace system for isolated KV storage
hr_namespace = db.create_namespace("dept.hr")
eng_namespace = db.create_namespace("dept.engineering")

# HR department stores its policies
hr_namespace.put(b"policy:pto", b"All employees receive 20 days PTO per year.")
hr_namespace.put(b"policy:parental_leave", b"16 weeks paid parental leave for all parents.")
print("HR facts stored in isolated namespace.")

# Engineering department stores its standards
eng_namespace.put(b"policy:primary_db", b"SochDB is the primary vector database for all AI services.")
eng_namespace.put(b"policy:deployment", b"All services deployed on Kubernetes via ArgoCD.")
print("Engineering facts stored in isolated namespace.")

# Verify isolation — HR data is only in HR namespace
hr_pto = hr_namespace.get(b"policy:pto")
print(f"\nHR PTO policy: {hr_pto.decode()}")

# Engineering cannot see HR data (different namespace)
eng_pto = eng_namespace.get(b"policy:pto")
print(f"Engineering sees HR PTO? {eng_pto}")  # Should be None — isolated!

HR facts stored in isolated namespace.
Engineering facts stored in isolated namespace.

HR PTO policy: All employees receive 20 days PTO per year.
Engineering sees HR PTO? None


### Step 4: Scan Namespace Data
Each namespace's data can be scanned independently — you only see what belongs to that namespace.

In [5]:
# Scan all policies in the HR namespace
hr_policies = hr_namespace.scan(b"policy:")
print("HR Namespace policies:")
for key, value in hr_policies:
    print(f"  {key.decode()} → {value.decode()}")

print()

# Scan all policies in the Engineering namespace
eng_policies = eng_namespace.scan(b"policy:")
print("Engineering Namespace policies:")
for key, value in eng_policies:
    print(f"  {key.decode()} → {value.decode()}")

HR Namespace policies:

Engineering Namespace policies:


### Step 5: Update Data Within a Namespace
Overwrite a policy in the HR namespace. Only HR data is affected.

In [6]:
# The PTO policy changed!
hr_namespace.put(b"policy:pto", b"All employees receive 25 days PTO per year, effective 2026.")
print("HR PTO policy updated.")

# Verify the update
updated_pto = hr_namespace.get(b"policy:pto")
print(f"Updated HR PTO: {updated_pto.decode()}")

# Engineering data is completely unaffected
eng_db_policy = eng_namespace.get(b"policy:primary_db")
print(f"\nEngineering DB policy (unchanged): {eng_db_policy.decode()}")

HR PTO policy updated.
Updated HR PTO: All employees receive 25 days PTO per year, effective 2026.

Engineering DB policy (unchanged): SochDB is the primary vector database for all AI services.


### Step 6: Cross-Namespace Grants
The `NamespaceManager` can create explicit, time-limited, auditable grants for cross-namespace access.

In [7]:
# Grant the Finance department read access to HR facts for 1 hour
grant = manager.create_grant(
    from_namespace="dept.finance",
    to_namespace="dept.hr",
    operations=["read"],
    expires_in_seconds=3600,
    reason="Finance needs HR policy data for budget planning"
)

print(f"Grant created: {grant.from_namespace} → {grant.to_namespace}")
print(f"Operations: {grant.operations}")
print(f"Reason: {grant.reason}")
print(f"Valid: {grant.is_valid()}")

Grant created: dept.finance → dept.hr
Operations: {'read'}
Reason: Finance needs HR policy data for budget planning
Valid: True


### Step 7: Namespace Metadata & Management
Inspect and manage namespace metadata programmatically.

In [8]:
# Check namespace existence
print(f"HR namespace exists: {manager.exists('dept.hr')}")
print(f"Sales namespace exists: {manager.exists('dept.sales')}")

# Get namespace metadata
hr_meta = manager.get_metadata("dept.hr")
print(f"\nHR metadata: {hr_meta}")

# Update metadata
manager.set_metadata("dept.hr", {"department": "Human Resources", "admin": "alice", "updated": "2026-03"})
print(f"Updated HR metadata: {manager.get_metadata('dept.hr')}")

# List all registered namespaces
all_namespaces = manager.list()
print(f"\nAll namespaces: {[str(ns) for ns in all_namespaces]}")

HR namespace exists: True
Sales namespace exists: False

HR metadata: {'department': 'Human Resources', 'admin': 'alice'}
Updated HR metadata: {'department': 'Human Resources', 'admin': 'alice', 'updated': '2026-03'}

All namespaces: ['dept.engineering', 'dept.finance', 'dept.hr']


### Cleanup

In [9]:
# Delete a namespace from the manager registry
manager.delete("dept.finance")
print(f"Finance namespace deleted. Remaining: {[str(ns) for ns in manager.list()]}")

# List SochDB-level namespaces
print(f"Database namespaces: {db.list_namespaces()}")

db.close()
print("Database closed.")

Finance namespace deleted. Remaining: ['dept.engineering', 'dept.hr']
Database namespaces: ['dept.engineering', 'dept.hr']
Database closed.
